# 05. Final Model and Evaluation

**Project:** Aura (Dormathon 2026, Track 1: Predict Early, Decide Better)

**Where we are:** notebook 04 chose gradient boosting, the 21 features, and an alert line of 30%. Every choice there was made on four fifths of the shipments. One fifth was locked away.

**Goal of this notebook:** open the locked test set once and answer the judging question directly: **does the model help someone make a better decision, and can we prove it?**

1. Train the final model.
2. Check it has learned patterns and not memorised shipments.
3. Score it on the test set: how many alerts are real, how many failures are caught.
4. Check the risk tiers and compare with the rule of thumb.
5. Show what drives its predictions.
6. Show that risk drops when a coordinator acts.
7. Stress it with noisy and missing input.
8. Save it for the app.

**The rule we follow:** nothing is tuned in this notebook. The model, features and alert line are fixed before the test set is opened, so the test numbers are a fair measure.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pio.renderers.default = "plotly_mimetype+notebook_connected"

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))  # lets us import our own code from src/

# Chart colours, kept the same in every chart
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb"


def style(fig, title, height=420):
    """Give every chart the same plain look."""
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=INK), x=0, xanchor="left"),
        height=height,
        paper_bgcolor=SURFACE,
        plot_bgcolor=SURFACE,
        font=dict(family="Segoe UI, Arial, sans-serif", size=12, color=MUTED),
        margin=dict(l=64, r=32, t=72, b=56),
        hoverlabel=dict(bgcolor="white", font_color=INK),
    )
    fig.update_xaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, linecolor=GRID, zeroline=False)
    return fig

## 1. Load the data and rebuild the same split

The features and the 80 / 20 split are identical to notebook 04, with the same random seed, so the test set here is exactly the fifth that notebook 04 never touched.

In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split

clean = pd.read_csv(ROOT / "data" / "processed" / "aura_shipments.csv")

SEED = 42
TARGET = "Shipment_Failure"

LEAKAGE_COLUMNS = ["Actual_Transit_Hours", "Late_Delivery", "Temperature_Excursion", "Handling_Damage"]


def add_features(df):
    """Add the engineered columns. Uses only inputs known during transit."""
    out = df.copy()
    eta = out["Transit_Duration_Hours"]            # current expected door-to-door time
    window = out["Lead_Time_Days"] * 24            # delivery window in hours
    plan = out["Planned_Transit_Hours"]
    buffer = out["Buffer_of_Package"]

    out["buffer_margin_hours"] = buffer - eta      # hours of cold left on arrival
    out["buffer_cover_ratio"] = buffer / eta
    out["schedule_slack_hours"] = window - eta     # hours it can still lose before it is late
    out["schedule_use_ratio"] = eta / window
    out["known_delay_hours"] = eta - plan          # delay already reported
    out["known_delay_ratio"] = eta / plan - 1
    out["tightest_margin_hours"] = np.minimum(out["buffer_margin_hours"], out["schedule_slack_hours"])
    out["buffer_short_flag"] = (out["buffer_margin_hours"] <= 0).astype(int)
    out["eta_past_window_flag"] = (out["schedule_slack_hours"] <= 0).astype(int)
    return out


RAW_NUMERIC = [
    "Distance_km", "Planned_Transit_Hours", "Transit_Duration_Hours", "Lead_Time_Days",
    "Buffer_of_Package", "Handover_Count", "Carrier_Reliability_Score",
    "Geopolitical_Risk_Score", "Fuel_Price_Index", "Stock_Volume",
]
ENGINEERED = [
    "buffer_margin_hours", "buffer_cover_ratio", "schedule_slack_hours", "schedule_use_ratio",
    "known_delay_hours", "known_delay_ratio", "tightest_margin_hours",
    "buffer_short_flag", "eta_past_window_flag",
]
MODEL_NUMERIC = RAW_NUMERIC + ENGINEERED
MODEL_CATEGORICAL = ["Transport_Mode", "Storage_standard"]
MODEL_FEATURES = MODEL_NUMERIC + MODEL_CATEGORICAL

# Decisions carried over from notebook 04. Nothing below changes them.
ALERT_LINE = 0.30
TIER_EDGES = [0, 0.30, 0.60, 0.80, 1.0001]
TIER_NAMES = ["Low (0 to 30%)", "Medium (31 to 60%)", "High (61 to 80%)", "Critical (above 80%)"]
MODEL_SETTINGS = dict(n_estimators=200, max_depth=2, learning_rate=0.03, subsample=0.8, random_state=SEED)

data = add_features(clean)
work, test = train_test_split(data, test_size=0.20, stratify=data[TARGET], random_state=SEED)
work, test = work.reset_index(drop=True), test.reset_index(drop=True)

X_work, y_work = work[MODEL_FEATURES], work[TARGET]
X_test, y_test = test[MODEL_FEATURES], test[TARGET]

assert not set(MODEL_FEATURES) & set(LEAKAGE_COLUMNS + [TARGET]), "An outcome column is in the feature list"
assert not set(work["Shipment_ID"]) & set(test["Shipment_ID"]), "A shipment is in both sets"

print(f"Working set : {len(work):,} shipments, {y_work.mean():.1%} fail")
print(f"Test set    : {len(test):,} shipments, {y_test.mean():.1%} fail")
print("No shipment appears in both sets, and no outcome column is used as an input.")

Working set : 4,000 shipments, 15.8% fail
Test set    : 1,000 shipments, 15.7% fail
No shipment appears in both sets, and no outcome column is used as an input.


## 2. Train the final model

The model is trained once on the whole working set, with the settings chosen in notebook 04.

One step is added in front of the model: if a number is missing, it is filled with the typical value for that column. The training data has no gaps, so this changes nothing here. It is there so that a shipment with a blank field does not crash the app. Section 7 tests how well that works.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def build_pipeline():
    numeric = Pipeline([("fill_missing", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    categorical = Pipeline([("fill_missing", SimpleImputer(strategy="most_frequent")),
                            ("encode", OneHotEncoder(handle_unknown="ignore"))])
    prepare = ColumnTransformer([("numeric", numeric, MODEL_NUMERIC), ("categorical", categorical, MODEL_CATEGORICAL)])
    return Pipeline([("prepare", prepare), ("model", GradientBoostingClassifier(**MODEL_SETTINGS))])


final_model = build_pipeline().fit(X_work, y_work)
print("Trained:", final_model.named_steps["model"])

Trained: GradientBoostingClassifier(learning_rate=0.03, max_depth=2, n_estimators=200,
                           random_state=42, subsample=0.8)


## 3. Has it learned, or memorised?

A model that memorised its training shipments scores far better on them than on new ones. We compare three scores:

- **Seen:** the working set it trained on.
- **Unseen, inside the working set:** 5-fold cross-validation, as in notebook 04.
- **Test:** the locked fifth, opened now for the first time.

If the three are close, the model has learned patterns that carry over to new shipments.

In [4]:
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import cross_val_predict

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
p_seen = final_model.predict_proba(X_work)[:, 1]
p_unseen = cross_val_predict(build_pipeline(), X_work, y_work, cv=folds, method="predict_proba")[:, 1]
p_test = final_model.predict_proba(X_test)[:, 1]


def ranking_scores(truth, predicted):
    return {"roc_auc": roc_auc_score(truth, predicted),
            "avg_precision": average_precision_score(truth, predicted),
            "brier": brier_score_loss(truth, predicted)}


memory_check = pd.DataFrame({
    "Seen (working set)": ranking_scores(y_work, p_seen),
    "Unseen (cross-validation)": ranking_scores(y_work, p_unseen),
    "Test set": ranking_scores(y_test, p_test),
}).T
memory_check["shipments"] = [len(y_work), len(y_work), len(y_test)]
print(f"Gap between seen and test, ROC AUC          : {memory_check.loc['Seen (working set)', 'roc_auc'] - memory_check.loc['Test set', 'roc_auc']:+.3f}")
print(f"Gap between seen and test, average precision: {memory_check.loc['Seen (working set)', 'avg_precision'] - memory_check.loc['Test set', 'avg_precision']:+.3f}")
memory_check.round(4)

Gap between seen and test, ROC AUC          : +0.048
Gap between seen and test, average precision: +0.072


,roc_auc,avg_precision,brier,shipments
Seen (working set),0.8845,0.7713,0.0661,4000
Unseen (cross-validation),0.8472,0.7229,0.0732,4000
Test set,0.8361,0.6993,0.0771,1000


**What this tells us:**

- **The three scores are close.** ROC AUC is 0.88 on shipments the model trained on, 0.85 in cross-validation and 0.84 on the test set.
- **There is a small gap, and it is the normal kind.** Every model does a little better on what it has seen. A gap of about 0.05 is mild. A model that had memorised its training shipments would show a far larger drop.
- **Notebook 04 predicted the test result well.** Cross-validation said 0.85 and the test set gave 0.84, inside the margin of error shown in the next section. The choices made in notebook 04 hold up on new shipments.

## 4. The scorecard on the test set

Every shipment at or above the 30% alert line is an alert. Four things can happen:

| | The shipment failed | The shipment was fine |
|---|---|---|
| **Alert raised** | Caught in time | False alarm |
| **No alert** | Missed | Correctly left alone |

In [5]:
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

alert = p_test >= ALERT_LINE
truth = y_test.to_numpy()
left_alone, false_alarm, missed, caught = confusion_matrix(truth, alert).ravel()

outcome_table = pd.DataFrame(
    {"The shipment failed": [caught, missed], "The shipment was fine": [false_alarm, left_alone]},
    index=["Alert raised", "No alert"],
)
print(f"Test shipments: {len(truth):,}   Failures: {truth.sum()}   Alerts raised: {alert.sum()}")
print()
print(f"Precision : {caught / (caught + false_alarm):.1%}   of {caught + false_alarm} alerts, {caught} were real")
print(f"Recall    : {caught / (caught + missed):.1%}   of {caught + missed} failures, {caught} were caught")
print(f"F1 score  : {f1_score(truth, alert):.3f}   (one number that balances precision and recall)")
print()
print(f"Accuracy                      : {(caught + left_alone) / len(truth):.1%}")
print(f"Accuracy of 'always say safe' : {1 - truth.mean():.1%}   (shown only to explain why we do not rely on accuracy)")
outcome_table

Test shipments: 1,000   Failures: 157   Alerts raised: 130

Precision : 70.0%   of 130 alerts, 91 were real
Recall    : 58.0%   of 157 failures, 91 were caught
F1 score  : 0.634   (one number that balances precision and recall)

Accuracy                      : 89.5%
Accuracy of 'always say safe' : 84.3%   (shown only to explain why we do not rely on accuracy)


,The shipment failed,The shipment was fine
Alert raised,91,39
No alert,66,804


### How sure are we about these numbers?

The test set has 1,000 shipments and about 157 failures. A different 1,000 would give slightly different scores. To see how much, we redraw the test set 2,000 times at random (with repeats allowed) and re-score each time. The range that holds 95% of those results is the honest margin of error.

In [6]:
rng = np.random.default_rng(SEED)
draws = []
for _ in range(2000):
    idx = rng.integers(0, len(truth), len(truth))
    t, a, p = truth[idx], alert[idx], p_test[idx]
    draws.append({
        "Precision": precision_score(t, a, zero_division=0), "Recall": recall_score(t, a),
        "F1 score": f1_score(t, a), "ROC AUC": roc_auc_score(t, p), "Average precision": average_precision_score(t, p),
    })
draws = pd.DataFrame(draws)

scorecard = pd.DataFrame({
    "test_score": {
        "Precision": precision_score(truth, alert), "Recall": recall_score(truth, alert), "F1 score": f1_score(truth, alert),
        "ROC AUC": roc_auc_score(truth, p_test), "Average precision": average_precision_score(truth, p_test),
    },
    "low_95": draws.quantile(0.025), "high_95": draws.quantile(0.975),
})

fig = go.Figure(go.Scatter(
    x=scorecard["test_score"], y=scorecard.index, mode="markers+text",
    marker=dict(color=BLUE, size=12, line=dict(color=SURFACE, width=2)),
    text=[f"{v:.2f}" for v in scorecard["test_score"]], textposition="top center", textfont=dict(color=INK),
    error_x=dict(type="data", symmetric=False, array=scorecard["high_95"] - scorecard["test_score"],
                 arrayminus=scorecard["test_score"] - scorecard["low_95"], color=BLUE, thickness=1.5, width=5),
    hovertemplate="%{y}: %{x:.3f}<extra></extra>",
))
fig.update_xaxes(range=[0.4, 1.0], title_text="Score on the test set (bar = 95% range)")
fig.update_yaxes(autorange="reversed")
style(fig, "Test-set scorecard at the 30% alert line", height=380)
fig.update_layout(margin=dict(l=150))
fig.show()

scorecard.round(3)

,test_score,low_95,high_95
Precision,0.700,0.619,0.782
Recall,0.580,0.503,0.656
F1 score,0.634,0.568,0.697
ROC AUC,0.836,0.793,0.876
Average precision,0.699,0.634,0.761


**What this tells us:**

- **Precision is 70%.** Of 130 alerts, 91 were real failures and 39 were false alarms. The 95% range is 62% to 78%.
- **Recall is 58%.** Of 157 failures, 91 were caught before they happened and 66 were missed. The 95% range is 50% to 66%.
- **These are a little lower than in notebook 04** (74% and 60%). That is expected, since notebook 04's settings were picked on the shipments it was scored on. Both notebook 04 figures sit inside the ranges above.
- **Accuracy is 89.5%**, against 84.3% for a model that never raises an alert. It is shown for completeness. The precision and recall figures are the ones that describe what a coordinator experiences.

**For the pitch:** "About 7 in 10 alerts are real, and Aura catches close to 6 in 10 failures before they happen."

## 5. Risk tiers and the rule of thumb, on the test set

Two checks from notebook 04 are repeated here on shipments the model has never seen.

**Do the tiers still mean what they say?**

In [7]:
tier = pd.cut(p_test, bins=TIER_EDGES, labels=TIER_NAMES, right=False)
tiers = pd.DataFrame({"failed": truth}).groupby(tier, observed=False).agg(
    shipments=("failed", "size"), failed=("failed", "sum"), failure_rate=("failed", "mean"))
tiers["share_of_all_failures"] = tiers["failed"] / tiers["failed"].sum()

fig = go.Figure(go.Bar(
    x=tiers.index.astype(str), y=tiers["failure_rate"], marker_color=BLUE,
    text=[f"{r:.0%}<br>{n:,} shipments" for r, n in zip(tiers["failure_rate"], tiers["shipments"])],
    textposition="outside", textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{x}<br>Really failed: %{y:.1%}<extra></extra>",
))
fig.update_yaxes(title_text="Share that really failed", tickformat=".0%", range=[0, 1.22])
style(fig, "What really happened in each risk tier (test set)", height=440)
fig.update_layout(bargap=0.4)
fig.show()

tiers.style.format({"failure_rate": "{:.1%}", "share_of_all_failures": "{:.1%}"})

,shipments,failed,failure_rate,share_of_all_failures
Low (0 to 30%),870,66,7.6%,42.0%
Medium (31 to 60%),52,20,38.5%,12.7%
High (61 to 80%),15,11,73.3%,7.0%
Critical (above 80%),63,60,95.2%,38.2%


**Is the model still better than the rule of thumb?** The rule ranks shipments by hours of margin left. Both get the same number of alerts.

In [8]:
value = test["Cargo_Value_USD"].to_numpy()
rule_score = -test["tightest_margin_hours"].to_numpy()
rule_alert = np.zeros(len(truth), dtype=bool)
rule_alert[np.argsort(-rule_score, kind="stable")[:alert.sum()]] = True


def alert_quality(flag, label):
    hit = flag & (truth == 1)
    return {"method": label, "alerts": int(flag.sum()), "real": int(hit.sum()),
            "false_alarms": int((flag & (truth == 0)).sum()),
            "precision": hit.sum() / flag.sum(), "recall": hit.sum() / truth.sum(),
            "cargo_value_in_caught_failures_usd": value[hit].sum()}


head_to_head = pd.DataFrame([alert_quality(rule_alert, "Rule of thumb"), alert_quality(alert, "Aura model")]).set_index("method")
fewer = 1 - head_to_head.loc["Aura model", "false_alarms"] / head_to_head.loc["Rule of thumb", "false_alarms"]
print(f"Same number of alerts ({alert.sum()}). The model raises {fewer:.0%} fewer false alarms than the rule of thumb.")

obvious = ((test["buffer_short_flag"] == 1) | (test["eta_past_window_flag"] == 1)).to_numpy()
hidden = ~obvious
print()
print(f"Shipments where nothing looks wrong yet: {hidden.sum():,}, of which {truth[hidden].mean():.1%} fail")
print(f"  model alerts among them : {(alert & hidden).sum()}, of which {truth[alert & hidden].mean():.0%} are real")
print(f"  not-obvious failures caught : {truth[alert & hidden].sum()} of {truth[hidden].sum()} ({truth[alert & hidden].sum() / truth[hidden].sum():.0%})")
print(f"  average precision, model against rule : {average_precision_score(truth[hidden], p_test[hidden]):.3f} against {average_precision_score(truth[hidden], rule_score[hidden]):.3f}")
head_to_head.style.format({"precision": "{:.0%}", "recall": "{:.0%}", "cargo_value_in_caught_failures_usd": "${:,.0f}"})

Same number of alerts (130). The model raises 15% fewer false alarms than the rule of thumb.

Shipments where nothing looks wrong yet: 968, of which 12.9% fail
  model alerts among them : 98, of which 60% are real
  not-obvious failures caught : 59 of 125 (47%)
  average precision, model against rule : 0.566 against 0.484


,alerts,real,false_alarms,precision,recall,cargo_value_in_caught_failures_usd
method,,,,,,
Rule of thumb,130,84,46,65%,54%,"$5,609,903"
Aura model,130,91,39,70%,58%,"$6,034,929"


**Which failures does it catch?** The target combines three kinds of failure. Here is the share of each kind that received an alert, and the same split by hospital urgency.

In [9]:
by_type = pd.DataFrame([
    {"failure_type": label, "failures_in_test": int(test[col].sum()),
     "caught": int((alert & (test[col] == 1)).sum()),
     "share_caught": (alert & (test[col] == 1)).sum() / test[col].sum()}
    for label, col in [("Late delivery", "Late_Delivery"), ("Temperature excursion", "Temperature_Excursion"),
                       ("Handling damage", "Handling_Damage")]
]).set_index("failure_type")

by_urgency = test.assign(alert=alert).groupby("Hospital_Urgency_Tier").apply(
    lambda g: pd.Series({
        "shipments": len(g), "failures": int(g[TARGET].sum()), "alerts": int(g["alert"].sum()),
        "precision": (g["alert"] & (g[TARGET] == 1)).sum() / max(g["alert"].sum(), 1),
        "recall": (g["alert"] & (g[TARGET] == 1)).sum() / max(g[TARGET].sum(), 1),
    }), include_groups=False)

display(by_type.style.format({"share_caught": "{:.0%}"}))
by_urgency.style.format({"shipments": "{:.0f}", "failures": "{:.0f}", "alerts": "{:.0f}", "precision": "{:.0%}", "recall": "{:.0%}"})

,failures_in_test,caught,share_caught
failure_type,,,
Late delivery,104,52,50%
Temperature excursion,84,53,63%
Handling damage,13,3,23%


,shipments,failures,alerts,precision,recall
Hospital_Urgency_Tier,,,,,
Critical Emergency,143,40,45,58%,65%
Elevated Procedure,306,55,42,81%,62%
Routine Restock,551,62,43,72%,50%


**What this tells us:**

- **The tiers hold on new shipments.** About 8% of Low shipments failed, 38% of Medium, 73% of High and 95% of Critical. Of 63 Critical shipments, 60 failed.
- **The model still beats the rule of thumb, by a smaller margin than before.** With 130 alerts each, the model caught 91 failures and the rule caught 84, which means 15% fewer false alarms. A difference of 7 shipments in a test of 1,000 is small enough that chance plays a part. The larger working set in notebook 04 showed 22%. The direction is the same in both, and the exact size is uncertain.
- **It helps where nothing looks wrong yet.** Among the 968 shipments with margin still left, the model caught 59 of 125 failures (47%), and 60% of its alerts there were real.
- **It is best at temperature excursions.** It caught 63% of excursions and 50% of late deliveries. It caught 3 of 13 handling-damage cases, which is no better than expected for an event that is close to random.
- **Critical Emergency shipments get the most protection and the most false alarms.** Recall is highest there (65%) and precision is lowest (58%). For the shipments a hospital can least afford to lose, erring towards an extra check is the safer side.

## 6. What drives the predictions?

A coordinator will not act on a number they do not understand. To see what the model relies on, we shuffle one feature at a time across the test shipments and measure how much worse the model gets. A feature the model leans on causes a big drop. A feature it ignores causes none.

Several features describe the same thing from different angles, so we also add them up by the question they answer.

In [10]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(final_model, X_test, y_test, scoring="average_precision",
                                    n_repeats=30, random_state=SEED)
drivers = pd.DataFrame({"drop_in_avg_precision": importance.importances_mean,
                        "spread": importance.importances_std}, index=MODEL_FEATURES)

QUESTION = {
    "Will the cold last?": ["buffer_margin_hours", "buffer_cover_ratio", "buffer_short_flag", "Buffer_of_Package"],
    "Will it arrive in time?": ["schedule_slack_hours", "schedule_use_ratio", "eta_past_window_flag", "Lead_Time_Days"],
    "Cold or clock, whichever is nearer": ["tightest_margin_hours"],
    "Is it already slipping?": ["known_delay_hours", "known_delay_ratio", "Transit_Duration_Hours", "Planned_Transit_Hours"],
    "Carrier and route": ["Carrier_Reliability_Score", "Geopolitical_Risk_Score", "Handover_Count", "Distance_km",
                          "Transport_Mode", "Storage_standard"],
    "Other": ["Fuel_Price_Index", "Stock_Volume"],
}
drivers["question"] = drivers.index.map({f: q for q, fs in QUESTION.items() for f in fs})

top = drivers.sort_values("drop_in_avg_precision", ascending=False).head(10).iloc[::-1]
fig = go.Figure(go.Bar(
    x=top["drop_in_avg_precision"], y=top.index, orientation="h", marker_color=BLUE,
    error_x=dict(type="data", array=top["spread"], color=INK, thickness=1, width=3),
    hovertemplate="%{y}<br>Drop in average precision: %{x:.3f}<extra></extra>",
))
fig.update_xaxes(title_text="Drop in average precision when this feature is shuffled")
style(fig, "The ten features the model leans on most (test set)", height=460)
fig.update_layout(margin=dict(l=190))
fig.show()

by_question = drivers.groupby("question")["drop_in_avg_precision"].sum().sort_values(ascending=False)
by_question.round(3).to_frame("total_drop_in_avg_precision")

,total_drop_in_avg_precision
question,
"Cold or clock, whichever is nearer",0.343
Will the cold last?,0.070
Will it arrive in time?,0.026
Carrier and route,0.017
Other,-0.000
Is it already slipping?,-0.003


**What this tells us:**

- **The model mostly asks one question: how many hours until the first limit is hit?** `tightest_margin_hours` matters several times more than anything else. This is the "hours of thermal runway" style number the triage queue shows, so what the coordinator sees on screen is what the model is reasoning from.
- **Then it adjusts.** How well the buffer covers the trip, how much of the delivery window is used, and how reliable the carrier is all move the prediction further.
- **The delay features score near zero here, and that needs explaining.** Delay still matters. A reported delay raises the ETA, and the ETA is already inside every margin. Shuffling the delay column alone changes little because the same information reaches the model another way.
- **`Fuel_Price_Index` and `Stock_Volume` contribute nothing.** They could be dropped from a future version without loss.

For the app, this supports explaining each alert with the three numbers a coordinator can check: hours of buffer margin, hours of schedule slack, and carrier reliability.

## 7. Does risk drop when the coordinator acts?

Prediction is only half of the product. The app lets a coordinator click an intervention and see the risk change. Here we test that loop on every alert in the test set.

**Three actions, each changing an input the coordinator really controls**

| Action | What changes | Assumed cost |
|---|---|---|
| Replenish dry ice | 24 more hours of buffer | $1,500 |
| Prioritise transfer | The delay reported so far is recovered, up to 6 hours | $2,500 |
| Both | Both changes | $4,000 |

The costs and the size of each effect are assumptions. The app shows them and lets the user change them.

**How money is counted:** expected loss is predicted risk times cargo value. The saving from an action is the drop in expected loss minus the cost of the action.

In [11]:
ACTIONS = {
    "No action": ({}, 0),
    "Replenish dry ice": ({"buffer_hours": 24}, 1_500),
    "Prioritise transfer": ({"recover_delay_hours": 6}, 2_500),
    "Both": ({"buffer_hours": 24, "recover_delay_hours": 6}, 4_000),
}
INPUT_COLUMNS = RAW_NUMERIC + MODEL_CATEGORICAL + ["Shipment_ID", "Cargo_Value_USD", "Hospital_Urgency_Tier"]


def apply_action(shipments, buffer_hours=0, recover_delay_hours=0):
    """Change the raw inputs the way the action would, then rebuild the features."""
    changed = shipments[INPUT_COLUMNS].copy()
    changed["Buffer_of_Package"] = changed["Buffer_of_Package"] + buffer_hours
    delay = (changed["Transit_Duration_Hours"] - changed["Planned_Transit_Hours"]).clip(lower=0)
    changed["Transit_Duration_Hours"] = changed["Transit_Duration_Hours"] - np.minimum(delay, recover_delay_hours)
    return add_features(changed)


alerted = test[alert].reset_index(drop=True)
risk_before = final_model.predict_proba(alerted[MODEL_FEATURES])[:, 1]
cargo = alerted["Cargo_Value_USD"].to_numpy()

rows, risk_after = [], {}
for name, (change, cost) in ACTIONS.items():
    risk = final_model.predict_proba(apply_action(alerted, **change)[MODEL_FEATURES])[:, 1]
    risk_after[name] = risk
    saving = (risk_before - risk) * cargo - cost
    rows.append({
        "action": name, "average_risk": risk.mean(),
        "alerts_brought_under_the_line": int((risk < ALERT_LINE).sum()),
        "share_brought_under_the_line": (risk < ALERT_LINE).mean(),
        "average_net_saving_usd": saving.mean(),
        "share_where_action_pays_off": (saving > 0).mean(),
    })
actions = pd.DataFrame(rows).set_index("action")

fig = go.Figure(go.Bar(
    x=actions.index, y=actions["average_risk"], marker_color=BLUE,
    text=[f"{r:.0%}" for r in actions["average_risk"]], textposition="outside",
    textfont=dict(color=INK), cliponaxis=False,
    hovertemplate="%{x}<br>Average predicted risk: %{y:.1%}<extra></extra>",
))
fig.add_hline(y=ALERT_LINE, line=dict(color=MUTED, width=1, dash="dash"), layer="below")
fig.add_annotation(xref="paper", x=1, y=ALERT_LINE, text="Alert line 30%", showarrow=False,
                   xanchor="right", yanchor="bottom", font=dict(color=INK))
fig.update_yaxes(title_text="Average predicted risk", tickformat=".0%", range=[0, 1.05])
style(fig, f"Predicted risk across the {len(alerted)} test alerts, before and after acting", height=420)
fig.update_layout(bargap=0.45)
fig.show()

actions.style.format({"average_risk": "{:.0%}", "share_brought_under_the_line": "{:.0%}",
                      "average_net_saving_usd": "${:,.0f}", "share_where_action_pays_off": "{:.0%}"})

,average_risk,alerts_brought_under_the_line,share_brought_under_the_line,average_net_saving_usd,share_where_action_pays_off
action,,,,,
No action,71%,0,0%,$0,0%
Replenish dry ice,44%,46,35%,"$16,704",68%
Prioritise transfer,32%,93,72%,"$25,250",80%
Both,13%,121,93%,"$36,086",97%


### One shipment, start to finish

This is the story the demo tells: the most urgent alert in the test set, ranked the way the triage queue ranks it (risk times cargo value times urgency weight, divided by hours of margin left).

In [12]:
URGENCY_WEIGHT = {"Routine Restock": 1.0, "Elevated Procedure": 1.5, "Critical Emergency": 2.5}

queue = alerted.assign(
    risk=risk_before,
    priority=risk_before * cargo * alerted["Hospital_Urgency_Tier"].map(URGENCY_WEIGHT)
    / alerted["tightest_margin_hours"].clip(lower=0.5),
)
# Pick the top shipment that can still be saved (some margin left), as the demo does
savable = queue[(queue["buffer_short_flag"] == 0) & (queue["eta_past_window_flag"] == 0)]
top = savable.sort_values("priority", ascending=False).iloc[0]
i = int(top.name)

print(f"Shipment        : {top['Shipment_ID']}   {top['Place_of_Origin']}  ->  {top['Place_of_Destination']}")
print(f"Cargo           : ${top['Cargo_Value_USD']:,.0f}, {top['Storage_standard']}, {top['Hospital_Urgency_Tier']}")
print(f"Buffer margin   : {top['buffer_margin_hours']:.1f} hours of cold left on arrival")
print(f"Schedule slack  : {top['schedule_slack_hours']:.1f} hours before the delivery window closes")
print(f"Delay reported  : {top['known_delay_hours']:.1f} hours, carrier reliability {top['Carrier_Reliability_Score']:.2f}")
print()
story = pd.DataFrame([
    {"action": name, "predicted_risk": risk_after[name][i],
     "expected_loss_usd": risk_after[name][i] * top["Cargo_Value_USD"],
     "action_cost_usd": cost,
     "net_saving_usd": (risk_before[i] - risk_after[name][i]) * top["Cargo_Value_USD"] - cost}
    for name, (_, cost) in ACTIONS.items()
]).set_index("action")
story.style.format({"predicted_risk": "{:.0%}", "expected_loss_usd": "${:,.0f}",
                    "action_cost_usd": "${:,.0f}", "net_saving_usd": "${:,.0f}"})

Shipment        : SC-12142   Los Angeles Pharmaceutical Logistics Depot  ->  Shanghai Medical Centre Depot
Cargo           : $113,972, 2°C to 8°C, Critical Emergency
Buffer margin   : 24.8 hours of cold left on arrival
Schedule slack  : 0.5 hours before the delivery window closes
Delay reported  : 9.7 hours, carrier reliability 0.58



,predicted_risk,expected_loss_usd,action_cost_usd,net_saving_usd
action,,,,
No action,98%,"$111,224",$0,$0
Replenish dry ice,97%,"$110,430","$1,500",$-706
Prioritise transfer,18%,"$20,996","$2,500","$87,728"
Both,18%,"$20,996","$4,000","$86,228"


**What this tells us:**

- **Acting lowers predicted risk.** Across the 130 test alerts the average risk is 71%. It falls to 44% with more dry ice, 32% with a prioritised transfer, and 13% with both.
- **Most actions pay for themselves under our assumptions.** Dry ice pays off for 68% of alerts, a prioritised transfer for 80%, and both together for 97%.
- **The model points to the right action, not just any action.** The example shipment has almost 25 hours of cold to spare but only half an hour of schedule slack. Adding dry ice leaves its risk at 97% and loses money, because cold is not its problem. Recovering the delay cuts the risk from 98% to 18% and protects about $88,000 of expected loss. A coordinator shown both options side by side can see which one is worth paying for.

**What this does and does not prove:** it shows that the model responds sensibly when an input changes, and in the direction a logistics expert would expect. It does not measure what an intervention achieves in the field. The 24 hours of added buffer, the 6 hours of recovered delay and the costs are assumptions.

## 8. Stress test: noisy and missing input

Sensor readings drift, ETAs are estimates, and forms are left half filled. The model has to stay useful when its inputs are imperfect.

### Noise

We add random error to every numeric input of every test shipment, rebuild the features, and re-score. A noise level of 10% means each value is typically off by 10% in either direction. Each level is repeated 30 times and averaged.

In [13]:
NOISY_COLUMNS = [c for c in RAW_NUMERIC if c != "Handover_Count"]   # a count of handovers is not a measurement
test_inputs = test[INPUT_COLUMNS]


def with_noise(shipments, level, rng):
    noisy = shipments.copy()
    for col in NOISY_COLUMNS:
        noisy[col] = noisy[col] * (1 + rng.normal(0, level, len(noisy)))
    noisy["Carrier_Reliability_Score"] = noisy["Carrier_Reliability_Score"].clip(0, 1)
    noisy["Geopolitical_Risk_Score"] = noisy["Geopolitical_Risk_Score"].clip(0, 10)
    for col in ["Distance_km", "Planned_Transit_Hours", "Transit_Duration_Hours", "Lead_Time_Days", "Buffer_of_Package"]:
        noisy[col] = noisy[col].clip(lower=0.1)
    return add_features(noisy)


def stress_scores(shipments):
    p = final_model.predict_proba(shipments[MODEL_FEATURES])[:, 1]
    flag = p >= ALERT_LINE
    return {"roc_auc": roc_auc_score(truth, p), "precision": precision_score(truth, flag, zero_division=0),
            "recall": recall_score(truth, flag), "alerts": flag.sum()}


rng = np.random.default_rng(SEED)
noise_rows = []
for level in [0.0, 0.02, 0.05, 0.10, 0.20]:
    runs = pd.DataFrame([stress_scores(with_noise(test_inputs, level, rng)) for _ in range(1 if level == 0 else 30)])
    noise_rows.append({"noise_level": level, **runs.mean().to_dict()})
noise = pd.DataFrame(noise_rows).set_index("noise_level")

fig = go.Figure()
for col, label, color in [("roc_auc", "ROC AUC", BLUE), ("precision", "Precision", ORANGE), ("recall", "Recall", AQUA)]:
    fig.add_trace(go.Scatter(x=noise.index, y=noise[col], mode="lines+markers", name=label,
                             line=dict(color=color, width=2), marker=dict(size=8, line=dict(color=SURFACE, width=2)),
                             hovertemplate=label + "<br>Noise %{x:.0%}<br>%{y:.3f}<extra></extra>"))
fig.update_xaxes(title_text="Noise added to every numeric input", tickformat=".0%", tickvals=list(noise.index))
fig.update_yaxes(range=[0, 1], title_text="Score on the test set")
style(fig, "How the scores hold up as the inputs get noisier", height=440)
fig.update_layout(legend=dict(orientation="h", y=-0.22, x=0, font=dict(color=INK)), margin=dict(b=100))
fig.show()

noise.round(3)

,roc_auc,precision,recall,alerts
noise_level,,,,
0.00,0.836,0.700,0.580,130.000
0.02,0.834,0.668,0.576,135.500
0.05,0.821,0.577,0.568,154.633
0.10,0.776,0.413,0.563,214.167
0.20,0.697,0.266,0.597,351.700


### Missing fields

Now we blank out one input at a time for every test shipment, as if that field were never supplied. The model fills the gap with a typical value and carries on. The table shows what that costs.

In [14]:
baseline = stress_scores(add_features(test_inputs))
missing_rows = [{"blank_field": "(nothing blank)", **baseline}]
for col in ["Buffer_of_Package", "Transit_Duration_Hours", "Lead_Time_Days", "Planned_Transit_Hours",
            "Carrier_Reliability_Score", "Geopolitical_Risk_Score", "Handover_Count", "Distance_km",
            "Fuel_Price_Index", "Stock_Volume"]:
    blanked = test_inputs.copy()
    blanked[col] = np.nan
    missing_rows.append({"blank_field": col, **stress_scores(add_features(blanked))})

missing = pd.DataFrame(missing_rows).set_index("blank_field")
missing["recall_lost"] = baseline["recall"] - missing["recall"]
missing.sort_values("recall", ascending=False).style.format(
    {"roc_auc": "{:.3f}", "precision": "{:.0%}", "recall": "{:.0%}", "alerts": "{:.0f}", "recall_lost": "{:+.0%}"})

,roc_auc,precision,recall,alerts,recall_lost
blank_field,,,,,
(nothing blank),0.836,70%,58%,130,+0%
Fuel_Price_Index,0.836,71%,58%,129,+0%
Planned_Transit_Hours,0.838,72%,58%,127,+0%
Distance_km,0.836,70%,58%,130,+0%
Handover_Count,0.833,72%,58%,126,+0%
Geopolitical_Risk_Score,0.834,70%,57%,128,+1%
Stock_Volume,0.836,70%,57%,129,+1%
Carrier_Reliability_Score,0.825,73%,54%,117,+4%
Lead_Time_Days,0.715,90%,30%,52,+28%


**What this tells us about noise:**

- **Small errors are absorbed.** At 2% noise the scores barely move.
- **Larger errors cause false alarms, not missed failures.** Recall stays near 57% at every noise level. Precision falls to 58% at 5% noise, 41% at 10% and 27% at 20%.
- **Why:** the model leans on margins, and a margin is a small difference between two large numbers. A shipment with a 40-hour buffer and a 37-hour ETA has 3 hours to spare. A 5% error on either input moves that margin by about 2 hours.
- **What to do about it:** the three time inputs (ETA, buffer and delivery window) need to be accurate to within about 5% for the alert line to hold at 30%. With noisier feeds, the line should be raised. This is a real limit and we state it.

**What this tells us about missing fields:**

- **Seven of the ten inputs can be blank with almost no loss.** Recall drops by 1 point or less.
- **A missing carrier reliability score costs about 4 points of recall.**
- **Three fields are essential.** Without the delivery window, recall falls to 30%. Without the buffer, to 11%. Without the ETA, the model raises no alerts at all.
- **What to do about it:** the app must not score a shipment that is missing its ETA, buffer or delivery window. It should show "Insufficient data for reliable prediction" instead of a low risk number that looks reassuring. These three fields are stored in the saved model as `required_fields`.

## 9. Save the model for the app

The file holds everything the app needs to score a shipment the same way this notebook does: the trained model, the feature list, the alert line, the tier boundaries and the test scores.

In [15]:
import joblib
import sklearn

MODEL_PATH = ROOT / "models" / "aura_final_model.pkl"
MODEL_PATH.parent.mkdir(exist_ok=True)

package = {
    "pipeline": final_model,
    "features": MODEL_FEATURES,
    "numeric_features": MODEL_NUMERIC,
    "categorical_features": MODEL_CATEGORICAL,
    "alert_line": ALERT_LINE,
    "tier_edges": TIER_EDGES,
    "tier_names": TIER_NAMES,
    "required_fields": ["Buffer_of_Package", "Transit_Duration_Hours", "Lead_Time_Days"],
    "test_scores": {k: round(float(v), 4) for k, v in scorecard["test_score"].items()},
    "trained_on_shipments": int(len(work)),
    "tested_on_shipments": int(len(test)),
    "sklearn_version": sklearn.__version__,
}
joblib.dump(package, MODEL_PATH, compress=3)

reloaded = joblib.load(MODEL_PATH)
same = np.allclose(reloaded["pipeline"].predict_proba(X_test)[:, 1], p_test)
print(f"Saved to {MODEL_PATH.relative_to(ROOT)} ({MODEL_PATH.stat().st_size / 1024:.0f} KB)")
print(f"Reloaded model gives identical predictions: {same}")
print("Stored test scores:", reloaded["test_scores"])

Saved to models\aura_final_model.pkl (47 KB)
Reloaded model gives identical predictions: True
Stored test scores: {'Precision': 0.7, 'Recall': 0.5796, 'F1 score': 0.6341, 'ROC AUC': 0.8361, 'Average precision': 0.6993}


**Using it:** build the features with `add_features`, then call `package["pipeline"].predict_proba(shipments[package["features"]])[:, 1]` to get the failure risk for each shipment.

The saved file is separate from `models/aura_model.pkl`, which `src/model.py` creates with the earlier feature set. Connecting the app to this model means moving `add_features` into `src/features.py` and updating the feature list in `src/schema.py`.

## The scorecard

All figures are from the 1,000 test shipments, which played no part in building the model.

| Measure | Result | 95% range |
|---|---|---|
| Precision (alerts that are real) | 70% | 62% to 78% |
| Recall (failures caught) | 58% | 50% to 66% |
| F1 score | 0.63 | 0.57 to 0.70 |
| ROC AUC | 0.84 | 0.79 to 0.88 |
| False alarms per 1,000 shipments | 39 | |
| Failure rate in the Critical tier | 95% | |
| Fewer false alarms than the rule of thumb, same workload | 15% | |

### Does it help someone make a better decision?

| What a coordinator needs | Evidence in this notebook |
|---|---|
| **Detect:** will it fail? | 7 in 10 alerts are real, and close to 6 in 10 failures are caught ahead of time (section 4) |
| **Categorize:** why? | Buffer margin and schedule slack show whether the cold or the clock is the problem (sections 6 and 7) |
| **Prioritize:** which first? | Risk tiers separate cleanly, from 8% in Low to 95% in Critical (section 5) |
| **Explain:** what is driving it? | The model leans on hours of margin, the same number shown on screen (section 6) |
| **Decide:** what to do? | The right action cuts risk sharply and the wrong one does not, with the saving shown in dollars (section 7) |

### What the model cannot do

- It misses about 4 in 10 failures. Most of those are caused by events that had not happened when the prediction was made.
- It raises more false alarms when its time inputs are off by more than about 5%.
- It cannot score a shipment without an ETA, a buffer and a delivery window.

### Limits we state openly

- The test set has 1,000 shipments, so every score carries the margin of error shown in section 4.
- The scores describe this dataset. Shipments from a different carrier or region could score differently, and the model should be re-checked on them before use.
- The intervention results show how the model's prediction responds when inputs change. The size of each effect and its cost are assumptions, and they should be replaced with a carrier's own figures.
- Handling damage is rare and close to random in this data, and the model should not be relied on to predict it.